# Узкое место Карты Знаний

## План устранения узкгого места

Вручную подобрать грамматические правила извлечения данных из текста, сравнивая их с каноничным вариантом статьи, то есть каким он должен получиться. Проверить их на корпусной лингвисики.

## Получение данных

Подключение к данным и сервисам

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "api"))

from api.services.nlp_grpc_client import get_nlp_grpc_client

nlp_client = get_nlp_grpc_client()
await nlp_client.connect()
print("NLP service connected on port 50055")

NLP service connected on port 50055


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "api"))

from neomodel import config as neomodel_config
from api.infrastructure.config import settings

database_url = settings.get_database_url()
neomodel_config.DATABASE_URL = database_url
if not settings.NEO4J_URI.startswith(("bolt+s://", "neo4j+s://")):
    neomodel_config.ENCRYPTED = False

print(f"Connected to Neo4j at {settings.NEO4J_URI}")

Connected to Neo4j at bolt://127.0.0.1:7687


C:\Users\dimka\AppData\Local\Temp\ipykernel_27080\2971084238.py:10: DeprecationWarning: Setting config.DATABASE_URL is deprecated and will be removed in a future version. Use the modern configuration API instead: from neomodel import get_config; config = get_config(); config.database_url = value
  neomodel_config.DATABASE_URL = database_url
C:\Users\dimka\AppData\Local\Temp\ipykernel_27080\2971084238.py:12: DeprecationWarning: Setting config.ENCRYPTED is deprecated and will be removed in a future version. Use the modern configuration API instead: from neomodel import get_config; config = get_config(); config.encrypted = value
  neomodel_config.ENCRYPTED = False


Визуализация схемы базы данных Neo4j

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "api"))

from neo4j import GraphDatabase
from api.infrastructure.config import settings
import plotly.graph_objects as go
import networkx as nx
import math

uri = settings.NEO4J_URI
user = settings.NEO4J_USER
password = settings.NEO4J_PASSWORD

driver = GraphDatabase.driver(uri, auth=(user, password))

with driver.session() as session:
    labels = [rec["label"] for rec in session.run("CALL db.labels()")]

labels_with_counts = []
for label in labels:
    with driver.session() as session:
        cnt = session.run(f"MATCH (n:`{label}`) RETURN count(*) AS cnt").single()["cnt"]
        labels_with_counts.append((label, cnt))

with driver.session() as session:
    rel_types = [rec["relationshipType"] for rec in session.run("CALL db.relationshipTypes()")]

edges = []
for rel_type in rel_types:
    with driver.session() as session:
        query = f"""
            MATCH (a)-[r:`{rel_type}`]->(b)
            WITH labels(a) AS src_labels, labels(b) AS dst_labels
            RETURN DISTINCT src_labels, dst_labels
        """
        for rec in session.run(query):
            src = rec["src_labels"][0] if rec["src_labels"] else None
            dst = rec["dst_labels"][0] if rec["dst_labels"] else None
            if src and dst:
                edges.append((src, dst, rel_type))

driver.close()

print(f"Labels in DB: {len(labels_with_counts)}")
for l, c in labels_with_counts:
    print(f"  {l}: {c}")
print(f"Relationship types: {len(rel_types)}")
for rt in rel_types:
    print(f"  {rt}")
print(f"Schema edges: {len(edges)}")
for s, d, r in edges:
    print(f"  {s} -[{r}]-> {d}")
print()

G = nx.DiGraph()

for label, cnt in labels_with_counts:
    G.add_node(label, count=cnt)

for src, dst, rel_type in edges:
    if G.has_edge(src, dst):
        G[src][dst]["label"] += f", {rel_type}"
    else:
        G.add_edge(src, dst, label=rel_type)

pos = nx.spring_layout(G, seed=42, k=2.5)

edge_traces = []
for src, dst, data in G.edges(data=True):
    x0, y0 = pos[src]
    x1, y1 = pos[dst]
    mid_x = (x0 + x1) / 2
    mid_y = (y0 + y1) / 2

    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode="lines",
        line=dict(width=1.5, color="#888"),
        hoverinfo="none",
        showlegend=False
    ))

    edge_traces.append(go.Scatter(
        x=[mid_x], y=[mid_y],
        mode="text",
        text=[data["label"]],
        textfont=dict(size=9, color="#555"),
        hoverinfo="none",
        showlegend=False
    ))

node_x, node_y, node_text, node_hover, node_size = [], [], [], [], []

for node in G.nodes():
    x, y = pos[node]
    cnt = G.nodes[node].get("count", 0)
    node_x.append(x)
    node_y.append(y)
    node_text.append(node)
    node_hover.append(f"{node}: {cnt} nodes")
    node_size.append(20 + math.log1p(cnt) * 3)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="top center",
    textfont=dict(size=11),
    hovertext=node_hover,
    hoverinfo="text",
    marker=dict(
        size=node_size,
        color="#4C9BE8",
        line=dict(width=2, color="white"),
    ),
    showlegend=False
)

fig = go.Figure(
    data=edge_traces + [node_trace],
    layout=go.Layout(
        height=700,
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        hovermode="closest",
        margin=dict(l=20, r=20, t=40, b=20),
        title=dict(text="Neo4j Schema Graph", font=dict(size=16)),
    )
)

fig.show()

Labels in DB: 5
  DataSource: 2
  Article: 209765
  Document: 13978
  MarkdownAnnotation: 1288725
  Action: 124
Relationship types: 5
  BIBLIOGRAPHIC_LINK
  HAS_MARKDOWN_ANNOTATION
  RELATES_TO
  LEADS_TO
  SYNTACTIC_DEP
Schema edges: 5
  Article -[BIBLIOGRAPHIC_LINK]-> Article
  Document -[HAS_MARKDOWN_ANNOTATION]-> MarkdownAnnotation
  MarkdownAnnotation -[RELATES_TO]-> MarkdownAnnotation
  Action -[LEADS_TO]-> Action
  Action -[SYNTACTIC_DEP]-> Action



Разметка spaCy всех документов в базе данных Neo4j.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "api"))

import uuid
import asyncio
from tqdm.notebook import tqdm
from neo4j import GraphDatabase
from api.infrastructure.config import settings
from api.services.nlp_grpc_client import get_nlp_grpc_client
from api.infrastructure.s3.s3_storage import get_s3_client

BUCKET = "knowledge-map-data"
uri = settings.NEO4J_URI
user = settings.NEO4J_USER
password = settings.NEO4J_PASSWORD

In [21]:
driver = GraphDatabase.driver(uri, auth=(user, password))

with driver.session() as session:
    docs = list(session.run("""
        MATCH (d:Document)
        WHERE (d.is_processed <> true OR d.is_processed IS NULL)
          AND (d.processing_status <> 'completed' OR d.processing_status IS NULL)
        RETURN d.uid AS uid, d.title AS title,
               d.s3_key AS s3_key
    """))

print(f"Unprocessed documents: {len(docs)}")
driver.close()


Unprocessed documents: 13978


In [22]:
from api.services.nlp_grpc_client import get_nlp_grpc_client
from api.infrastructure.s3.s3_storage import get_s3_client

nlp_client = get_nlp_grpc_client()
await nlp_client.connect()
s3_client = get_s3_client()

driver = GraphDatabase.driver(uri, auth=(user, password))
pbar = tqdm(docs)

for doc_rec in pbar:
    uid = doc_rec["uid"]
    md_key = doc_rec["s3_key"]

    with driver.session() as session:
        if not md_key:
            session.run("MATCH (d:Document {uid: $uid}) SET d.processing_status = 'error', d.error_message = 'no S3 key'", uid=uid)
            continue
        session.run("MATCH (d:Document {uid: $uid}) SET d.processing_status = 'processing'", uid=uid)

    try:
        text = await s3_client.download_text(BUCKET, md_key)
        if not text:
            with driver.session() as session:
                session.run("MATCH (d:Document {uid: $uid}) SET d.processing_status = 'error', d.error_message = 'empty text'", uid=uid)
            continue

        result = await nlp_client.process_text(text=text, merge_results=True)
        if not result.get("success"):
            with driver.session() as session:
                session.run("MATCH (d:Document {uid: $uid}) SET d.processing_status = 'error', d.error_message = $msg", uid=uid, msg=result.get("message", "unknown"))
            continue

        merged = result.get("merged_result", result)
        annotations = merged.get("annotations", [])

        with driver.session() as session:
            if annotations:
                rows = [{
                    "uid": str(uuid.uuid4()),
                    "text": a.get("text", ""),
                    "annotation_type": a.get("annotation_type", ""),
                    "start_offset": a.get("start_offset", 0),
                    "end_offset": a.get("end_offset", 0),
                    "confidence": a.get("confidence", 0.0),
                    "source": str(a.get("source", "spacy")),
                } for a in annotations]

                session.run("""
                    UNWIND $rows AS row
                    CREATE (a:MarkdownAnnotation {
                        uid: row.uid, text: row.text,
                        annotation_type: row.annotation_type,
                        start_offset: row.start_offset,
                        end_offset: row.end_offset,
                        confidence: row.confidence,
                        source: row.source,
                        created_date: datetime()
                    })
                    WITH a
                    MATCH (d:Document {uid: $uid})
                    CREATE (d)-[:HAS_MARKDOWN_ANNOTATION]->(a)
                """, uid=uid, rows=rows)

            session.run("""
                MATCH (d:Document {uid: $uid})
                SET d.is_processed = true,
                    d.processing_status = 'completed',
                    d.error_message = null
            """, uid=uid)

    except Exception as e:
        with driver.session() as session:
            session.run(
                "MATCH (d:Document {uid: $uid}) SET d.processing_status = 'error', d.error_message = $msg",
                uid=uid, msg=str(e)
            )

driver.close()
await nlp_client.disconnect()

NameError: name 'get_s3_client' is not defined

Получение данных одной статьи

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "api"))

import pandas as pd
from neo4j import GraphDatabase
from api.infrastructure.config import settings

# UID статьи для проверки
ARTICLE_UID = "PMC176545"

uri = settings.NEO4J_URI
user = settings.NEO4J_USER
password = settings.NEO4J_PASSWORD

driver = GraphDatabase.driver(uri, auth=(user, password))

with driver.session() as session:
    # Информация о документе
    doc_rec = session.run(
        "MATCH (d:Document {uid: $uid}) RETURN d",
        uid=ARTICLE_UID
    ).single()

    if not doc_rec:
        print(f"Document {ARTICLE_UID} not found")
    else:
        d = dict(doc_rec["d"])
        print("=== Document ===")
        for k, v in d.items():
            print(f"  {k}: {v}")

        # Все аннотации разметки
        ann_records = list(session.run(
            """
            MATCH (d:Document {uid: $uid})-[:HAS_MARKDOWN_ANNOTATION]->(a:MarkdownAnnotation)
            RETURN a.uid AS uid, a.text AS text,
                   a.annotation_type AS type,
                   a.start_offset AS start_offset,
                   a.end_offset AS end_offset,
                   a.confidence AS confidence,
                   a.source AS source
            ORDER BY a.start_offset
            """,
            uid=ARTICLE_UID
        ))

        print(f"\n=== Annotations: {len(ann_records)} ===")
        df = pd.DataFrame([dict(r) for r in ann_records])
        if not df.empty:
            display(df.head(20))
            print(f"... total {len(df)} annotations")
        else:
            print("No annotations found")

driver.close()

=== Document ===
  is_processed: True
  md5_hash: pmc_PMC176545
  keywords: []
  is_open_access: True
  abstract: Plasmodium falciparum is the causative agent of the most burdensome form of human malaria, affecting 200–300 million individuals per year worldwide. The recently sequenced genome of P. falciparum revealed over 5,400 genes, of which 60% encode proteins of unknown function. Insights into the biochemical function and regulation of these genes will provide the foundation for future drug and vaccine development efforts toward eradication of this disease. By analyzing the complete asexual intraerythrocytic developmental cycle (IDC) transcriptome of the HB3 strain of P. falciparum, we demonstrate that at least 60% of the genome is transcriptionally active during this stage. Our data demonstrate that this parasite has evolved an extremely specialized mode of transcriptional regulation that produces a continuous cascade of gene expression, beginning with genes corresponding to gener

,uid,text,type,start_offset,end_offset,confidence,source
0,0d01d90d-b094-4744-943f-6ac373958db9,#,PUNCT,0,1,1.0,multilevel_nlp
1,f47eb1bb-56ec-4bc3-8132-6bdcb9ea0db9,The,DET,2,5,1.0,multilevel_nlp
2,4577275d-21c1-48a2-91ff-18d2a1b6b93d,Transcriptome,ENTITY,6,19,1.0,SPACY
3,7ac0705a-27c9-4d11-a673-f4c3e0e95144,Transcriptome,ENTITY,6,19,1.0,multilevel_nlp
4,fef92028-921b-4701-860a-448dddf1ca55,Transcriptome,NOUN,6,19,1.0,multilevel_nlp
5,90fbeb19-7be4-4538-81fd-7a1b3f3149d7,of,ADP,20,22,1.0,multilevel_nlp
6,4a40b122-f1bd-48fb-bd6a-639e92c1a95a,the,DET,23,26,1.0,multilevel_nlp
7,3527c268-eaad-45a8-aed3-058de2b50ae1,Intraerythrocytic Developmental Cycle,ENTITY,27,64,1.0,SPACY
8,5d04b1bb-041e-4334-a93b-ab8ea441e2ef,Intraerythrocytic,ADJ,27,44,1.0,multilevel_nlp
9,c5a0d604-aa8e-4987-a167-7679721e1c11,Intraerythrocytic Developmental Cycle,ENTITY,27,64,1.0,multilevel_nlp


... total 18000 annotations


## Восстановление spaCy Doc из MarkdownAnnotation

Загружает аннотации и синтаксические связи (`RELATES_TO`) из Neo4j, восстанавливает spaCy `Doc` 
для использования с `Matcher` без повторного запуска spaCy-пайплайна.

In [8]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "api"))

import json
from spacy.matcher import Matcher
from spacy.tokens import Doc
from spacy.vocab import Vocab
from neo4j import GraphDatabase
from api.infrastructure.config import settings

ARTICLE_UID = "PMC176545"

uri = settings.NEO4J_URI
user = settings.NEO4J_USER
password = settings.NEO4J_PASSWORD
driver = GraphDatabase.driver(uri, auth=(user, password))

# Шаг 1a: загрузить токен-аннотации (с POS/леммой/морфологией)
anns = list(driver.execute_query(
    """
    MATCH (d:Document {uid: $uid})-[:HAS_MARKDOWN_ANNOTATION]->(a:MarkdownAnnotation)
    WHERE a.source = 'multilevel_nlp' AND a.annotation_type <> 'ENTITY'
    RETURN a.uid AS uid, a.text AS text, a.annotation_type AS ann_type,
           a.start_offset AS start_offset, a.end_offset AS end_offset,
           a.metadata AS metadata_json
    """, uid=ARTICLE_UID
)[0])
anns = [dict(r) for r in anns]
print(f"Загружено токен-аннотаций: {len(anns)}")

# Парсим metadata и сортируем
for r in anns:
    r["meta"] = json.loads(r["metadata_json"])
anns.sort(key=lambda r: (r["meta"].get("sent_idx", 0), r["meta"].get("token_idx", 0)))

# Шаг 1b: загрузить ENTITY-аннотации (для ent_type)
entities = list(driver.execute_query(
    """
    MATCH (d:Document {uid: $uid})-[:HAS_MARKDOWN_ANNOTATION]->(a:MarkdownAnnotation)
    WHERE a.source = 'multilevel_nlp' AND a.annotation_type = 'ENTITY'
    RETURN a.start_offset AS start_offset, a.end_offset AS end_offset,
           a.metadata AS metadata_json
    """, uid=ARTICLE_UID
)[0])
print(f"Загружено ENTITY-аннотаций: {len(entities)}")
ent_lookup = {}
for r_ent in entities:
    r_ent = dict(r_ent)
    m = json.loads(r_ent["metadata_json"])
    ent_type = m.get("entity_type", "ENTITY")
    ent_lookup[(r_ent["start_offset"], r_ent["end_offset"])] = ent_type

# Шаг 2: uid -> order_idx
uid_to_idx = {r["uid"]: i for i, r in enumerate(anns)}

# Шаг 3: загрузить RELATES_TO (dependency)
deps = list(driver.execute_query(
    """
    MATCH (a:MarkdownAnnotation)-[r:RELATES_TO]->(b:MarkdownAnnotation)
    WHERE a.uid IN $uids AND b.uid IN $uids
    RETURN a.uid AS src, b.uid AS tgt, r.relation_type AS dep
    """, uids=list(uid_to_idx.keys())
)[0])
deps = [dict(r) for r in deps]
print(f"Загружено зависимостей: {len(deps)}")

# Строим head_idx и dep_ для каждого токена
head_idx = [-1] * len(anns)
dep_tag = [""] * len(anns)
for d in deps:
    child = uid_to_idx.get(d["src"])
    parent = uid_to_idx.get(d["tgt"])
    if child is not None and parent is not None:
        head_idx[child] = parent
        dep_tag[child] = d["dep"]

# Шаг 4: восстановить spaces (is followed by space)
words = []
spaces = []
for i, r in enumerate(anns):
    words.append(r["text"])
    if i + 1 < len(anns):
        spaces.append(anns[i + 1]["start_offset"] > r["end_offset"])
    else:
        spaces.append(False)

# Шаг 5: создать spaCy Doc
vocab = Vocab()
doc = Doc(vocab, words=words, spaces=spaces)

for i, r in enumerate(anns):
    token = doc[i]
    meta = r["meta"]
    token.lemma_ = meta.get("lemma", r["text"])
    token.pos_ = meta.get("pos", "X")
    token.tag_ = meta.get("pos_fine", "")
    # Восстанавливаем ent_type из ENTITY-аннотаций по позиции токена
    ent = ent_lookup.get((r["start_offset"], r["end_offset"]))
    if ent:
        token.ent_type_ = ent
    if dep_tag[i]:
        token.dep_ = dep_tag[i]
    if head_idx[i] >= 0:
        token.head = doc[head_idx[i]]

print(f"spaCy Doc создан: {len(doc)} токенов, {len(list(doc.sents))} предложений")
print(f"  Текст (первые 200 символов): {doc.text[:200]}...\n")

# Шаг 6: тест Matcher
matcher = Matcher(vocab)
pattern = [
    {"POS": "DET"},
    {"POS": "NOUN"},
]
matcher.add("DET+NOUN", [pattern])
matches = matcher(doc)
print(f"Matcher нашёл {len(matches)} совпадений по шаблону DET+NOUN:")
for match_id, start, end in matches[:10]:
    span = doc[start:end]
    print(f"  {span.text}  (POS={[t.pos_ for t in span]})")

# Пример: найти фразы вида "NOUN of NOUN"
matcher2 = Matcher(vocab)
matcher2.add("NOUN_of_NOUN", [[{"POS": "NOUN"}, {"LEMMA": "of"}, {"POS": "NOUN"}]])
matches2 = matcher2(doc)
print(f"\nMatcher нашёл {len(matches2)} совпадений по шаблону NOUN of NOUN:")
for match_id, start, end in matches2[:10]:
    span = doc[start:end]
    print(f"  {span.text}")

driver.close()

Загружено токен-аннотаций: 11957
Загружено ENTITY-аннотаций: 2891
Загружено зависимостей: 11500
spaCy Doc создан: 11957 токенов, 1 предложений
  Текст (первые 200 символов): # The Transcriptome of the Intraerythrocytic Developmental Cycle of Plasmodium falciparum **Авторы:** Bozdech Zbynek, Llinás Manuel, Pulliam Brian Lee, Wong Edith D, Zhu Jingchun, DeRisi Joseph L **Жу...

Matcher нашёл 684 совпадений по шаблону DET+NOUN:
  The Transcriptome  (POS=['DET', 'NOUN'])
  these genes  (POS=['DET', 'NOUN'])
  the foundation  (POS=['DET', 'NOUN'])
  this disease  (POS=['DET', 'NOUN'])
  the HB3  (POS=['DET', 'NOUN'])
  the genome  (POS=['DET', 'NOUN'])
  this stage  (POS=['DET', 'NOUN'])
  this parasite  (POS=['DET', 'NOUN'])
  The data  (POS=['DET', 'NOUN'])
  the chromosomes  (POS=['DET', 'NOUN'])

Matcher нашёл 120 совпадений по шаблону NOUN of NOUN:
  cascade of gene
  regions of chromosomes
  cascade of gene
  timing of transcription
  episodes of malaria
  combinations of vector
  pau

## Грамматика для извлечения конструкций

- `a`, `an` — неопределённый артикаль
- `the` — определённый артикаль
- `is`
- `is a`, `is an`

In [ ]:
matcher_a = Matcher(vocab)
matcher_a.add("a/an {}", [[
    {"LEMMA": {"IN": ["a", "an"]}},
    {},
]])
matches_a = matcher_a(doc)
print(f"Matcher нашёл {len(matches_a)} совпадений")
for match_id, start, end in matches_a[:10]:
    span = doc[start:end]
    print(f"  {span.text}  (POS={[t.pos_ for t in span]})")

Matcher нашёл 203 совпадений
  an extremely  (POS=['DET', 'ADV'])
  a continuous  (POS=['DET', 'ADJ'])
  a “  (POS=['DET', 'PUNCT'])
  a time  (POS=['DET', 'NOUN'])
  a resource  (POS=['DET', 'NOUN'])
  a global  (POS=['DET', 'ADJ'])
  a worldwide  (POS=['DET', 'ADJ'])
  a malaria  (POS=['DET', 'NOUN'])
  a comprehensive  (POS=['DET', 'ADJ'])
  a circular  (POS=['DET', 'ADJ'])


In [ ]:
matcher_the = Matcher(vocab)
matcher_the.add("the {}", [[
    {"LEMMA": "the"},
    {},
]])
matches_the = matcher_the(doc)
print(f"Matcher нашёл {len(matches_the)} совпадений")
for match_id, start, end in matches_the[:10]:
    span = doc[start:end]
    print(f"  {span.text}  (POS={[t.pos_ for t in span]})")

Matcher нашёл 794 совпадений
  The Transcriptome  (POS=['DET', 'NOUN'])
  the Intraerythrocytic  (POS=['DET', 'ADJ'])
  the causative  (POS=['DET', 'ADJ'])
  the most  (POS=['DET', 'ADV'])
  The recently  (POS=['DET', 'ADV'])
  the biochemical  (POS=['DET', 'ADJ'])
  the foundation  (POS=['DET', 'NOUN'])
  the complete  (POS=['DET', 'ADJ'])
  the HB3  (POS=['DET', 'NOUN'])
  the genome  (POS=['DET', 'NOUN'])


In [26]:
matcher_be = Matcher(vocab)
matcher_be.add("be/is/was/were/are/am/been/being {}", [[
    {"LEMMA": "be"},
    {},
]])
matches_be = matcher_be(doc)
print(f"Matcher нашёл {len(matches_be)} совпадений")
for match_id, start, end in matches_be[:10]:
    span = doc[start:end]
    print(f"  {span.text}  (POS={[t.pos_ for t in span]})")

Matcher нашёл 367 совпадений
  is the  (POS=['AUX', 'DET'])
  is transcriptionally  (POS=['AUX', 'ADV'])
  are rarely  (POS=['AUX', 'ADV'])
  is highly  (POS=['AUX', 'ADV'])
  was used  (POS=['AUX', 'VERB'])
  were found  (POS=['AUX', 'VERB'])
  is unprecedented  (POS=['AUX', 'ADJ'])
  is required  (POS=['AUX', 'VERB'])
  is caused  (POS=['AUX', 'VERB'])
  is responsible  (POS=['AUX', 'ADJ'])


In [29]:
matcher_adj_dep_noun = Matcher(vocab)
matcher_adj_dep_noun.add("be/is/was/were/are/am/been/being {}", [[
    {"POS": "ADJ", "DEP": {}},
    {"POS": "NOUN"},
]])
matches_adj_dep_noun = matcher_adj_dep_noun(doc)
print(f"Matcher нашёл {len(matches_adj_dep_noun)} совпадений")
for match_id, start, end in matches_adj_dep_noun[:10]:
    span = doc[start:end]
    print(f"  {span.text}  (POS={[t.pos_ for t in span]}, DEP={[t.dep_ for t in span]}")

Matcher нашёл 816 совпадений
  Developmental Cycle  (POS=['ADJ', 'NOUN'], DEP=['amod', 'nmod']
  causative agent  (POS=['ADJ', 'NOUN'], DEP=['amod', '']
  burdensome form  (POS=['ADJ', 'NOUN'], DEP=['amod', 'nmod']
  human malaria  (POS=['ADJ', 'NOUN'], DEP=['amod', 'nmod']
  unknown function  (POS=['ADJ', 'NOUN'], DEP=['amod', 'nmod']
  biochemical function  (POS=['ADJ', 'NOUN'], DEP=['amod', 'nmod']
  future drug  (POS=['ADJ', 'NOUN'], DEP=['amod', 'compound']
  developmental cycle  (POS=['ADJ', 'NOUN'], DEP=['amod', 'compound']
  specialized mode  (POS=['ADJ', 'NOUN'], DEP=['amod', 'dobj']
  transcriptional regulation  (POS=['ADJ', 'NOUN'], DEP=['amod', 'nmod']


Вытаскивает все возможные n-граммы с синтаксическими связями и колокации.

In [ ]:
from collections import Counter, defaultdict
import pandas as pd
import json

ARTICLE_UID = "PMC176545"
SHOW_TOP = 3000
MIN_N = 2
MAX_N = 10
MIN_FREQ = 2

aid_to_idx = {r["uid"]: i for i, r in enumerate(anns)}
child_to_heads = defaultdict(list)
for d in deps:
    src = aid_to_idx.get(d["src"])
    tgt = aid_to_idx.get(d["tgt"])
    if src is not None and tgt is not None:
        child_to_heads[src].append((tgt, d["dep"]))

for child, head_infos in child_to_heads.items():
    for head_idx, dep in head_infos:
        doc[child].dep_ = dep
        doc[child].head = doc[head_idx]

data = Counter()
for n in range(MIN_N, MAX_N + 1):
    for i in range(len(doc) - n + 1):
        span = doc[i:i+n]
        if any(t.is_punct for t in span):
            continue
        text = " ".join(t.text for t in span)
        idx_set = {t.i for t in span}
        tokens_info = []
        edges = []
        for t in span:
            tokens_info.append({"pos": t.pos_, "dep": t.dep_ if t.dep_ else "?", "text": t.text})
            if t.i in child_to_heads:
                for head_idx, dep in child_to_heads[t.i]:
                    if head_idx in idx_set:
                        edges.append({
                            "from_pos": t.pos_, "from_text": t.text,
                            "dep": dep,
                            "to_pos": doc[head_idx].pos_, "to_text": doc[head_idx].text,
                        })
        graph = {"tokens": tokens_info, "edges": edges}
        data[(text, json.dumps(graph, ensure_ascii=False))] += 1

if MIN_FREQ > 1:
    data = Counter({k: v for k, v in data.items() if v >= MIN_FREQ})

df = pd.DataFrame([
    {"n-gram": k[0], "pattern": k[1], "count": v}
    for k, v in sorted(data.items(), key=lambda x: -x[1])[:SHOW_TOP]
])
df

,n-gram,pattern,count
0,of the,"{""tokens"": [{""pos"": ""ADP"", ""dep"": ""case"", ""tex...",168
1,) .,"{""tokens"": [{""pos"": ""PUNCT"", ""dep"": ""punct"", ""...",131
2,. The,"{""tokens"": [{""pos"": ""PUNCT"", ""dep"": ""punct"", ""...",94
3,in the,"{""tokens"": [{""pos"": ""ADP"", ""dep"": ""case"", ""tex...",72
4,", and","{""tokens"": [{""pos"": ""PUNCT"", ""dep"": ""punct"", ""...",54
...,...,...,...
2995,every timepoint . The 2-h,"{""tokens"": [{""pos"": ""DET"", ""dep"": ""det"", ""text...",2
2996,timepoint . The 2-h invasion,"{""tokens"": [{""pos"": ""NOUN"", ""dep"": ""nmod"", ""te...",2
2997,. The 2-h invasion window,"{""tokens"": [{""pos"": ""PUNCT"", ""dep"": ""punct"", ""...",2
2998,The 2-h invasion window during,"{""tokens"": [{""pos"": ""DET"", ""dep"": ""det"", ""text...",2
